In [1]:
# DATA ANALYST INTERNSHIP - TASK 1
# Data Cleaning and Preprocessing

import pandas as pd
import numpy as np
import os

In [6]:
# 1. Load dataset
from google.colab import files

uploaded = files.upload()

file_path = next(iter(uploaded))
df = pd.read_csv(file_path, sep="\t")


print("========== ORIGINAL DATASET ==========")
print("Original shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nMissing values before cleaning:")
print(df.isnull().sum())
print("\nDuplicate rows before cleaning:", df.duplicated().sum())

Saving marketing_campaign.csv to marketing_campaign.csv
========== ORIGINAL DATASET ==========
Original shape: (2240, 29)

First 5 rows:


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0



Missing values before cleaning:
ID                      0
Year_Birth              0
Education               0
Marital_Status          0
Income                 24
Kidhome                 0
Teenhome                0
Dt_Customer             0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Complain                0
Z_CostContact           0
Z_Revenue               0
Response                0
dtype: int64

Duplicate rows before cleaning: 0


In [7]:
# 2. Standardize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [8]:
# 3. Remove duplicate records
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# 4. Clean text columns
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df[col] = df[col].str.strip()

# Standardize categorical values
for col in ["education", "marital_status"]:
    if col in df.columns:
        df[col] = df[col].str.title()



In [9]:
# 5. Handle missing income values
income_missing_filled = 0

if "income" in df.columns:
    df["income"] = pd.to_numeric(df["income"], errors="coerce")
    income_missing_filled = int(df["income"].isnull().sum())

    if df["income"].notnull().any():
        df["income"] = df["income"].fillna(df["income"].median())



In [10]:
# 6. Convert date column
if "dt_customer" in df.columns:
    df["dt_customer"] = pd.to_datetime(
        df["dt_customer"],
        format="%d-%m-%Y",
        errors="coerce"
    )



In [11]:
# 7. Correct year of birth data type
if "year_birth" in df.columns:
    df["year_birth"] = pd.to_numeric(
        df["year_birth"], errors="coerce"
    ).astype("Int64")



In [12]:
# 8. Identify potential income outliers using IQR
outlier_count = 0

if "income" in df.columns and df["income"].notnull().sum() >= 2:
    q1 = df["income"].quantile(0.25)
    q3 = df["income"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = int(
        ((df["income"] < lower_bound) |
         (df["income"] > upper_bound)).sum()
    )


In [13]:
# 9. Validate data quality
missing_after = int(df.isnull().sum().sum())
duplicates_after = int(df.duplicated().sum())

print("\n========== CLEANING RESULTS ==========")
print("Duplicate rows removed:", duplicates_removed)
print("Missing income values filled:", income_missing_filled)
print("Potential income outliers identified:", outlier_count)
print("Remaining missing values:", missing_after)
print("Remaining duplicate rows:", duplicates_after)
print("Final dataset shape:", df.shape)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nFinal data types:")
print(df.dtypes)




========== CLEANING RESULTS ==========
Duplicate rows removed: 0
Missing income values filled: 24
Potential income outliers identified: 8
Remaining missing values: 0
Remaining duplicate rows: 0
Final dataset shape: (2240, 29)

Missing values after cleaning:
id                     0
year_birth             0
education              0
marital_status         0
income                 0
kidhome                0
teenhome               0
dt_customer            0
recency                0
mntwines               0
mntfruits              0
mntmeatproducts        0
mntfishproducts        0
mntsweetproducts       0
mntgoldprods           0
numdealspurchases      0
numwebpurchases        0
numcatalogpurchases    0
numstorepurchases      0
numwebvisitsmonth      0
acceptedcmp3           0
acceptedcmp4           0
acceptedcmp5           0
acceptedcmp1           0
acceptedcmp2           0
complain               0
z_costcontact          0
z_revenue              0
response               0
dtype: int64

Fi

In [14]:
# 10. Save cleaned dataset
os.makedirs("output", exist_ok=True)

df.to_csv(
    "output/cleaned_marketing_campaign.csv",
    index=False,
    date_format="%d-%m-%Y"
)



In [16]:
from google.colab import files

files.download("output/cleaned_marketing_campaign.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# 11. Generate cleaning summary report
summary = f"""# Data Cleaning and Preprocessing Report

## Dataset
Customer Personality Analysis

## Tools
Python, Pandas, NumPy, Jupyter Notebook

## Cleaning Operations
1. Inspected dataset structure and missing values.
2. Standardized column names.
3. Removed duplicate records.
4. Trimmed whitespace and standardized selected categories.
5. Filled missing income values using the median, when available.
6. Converted customer enrollment dates to datetime.
7. Corrected the year of birth data type.
8. Identified potential income outliers using the IQR method.
9. Validated the final dataset.

## Results
- Original rows: {original_rows if False else "See notebook output"}
- Final rows: {len(df)}
- Duplicate rows removed: {duplicates_removed}
- Missing income values filled: {income_missing_filled}
- Potential income outliers identified: {outlier_count}
- Remaining missing values: {missing_after}
- Remaining duplicate rows: {duplicates_after}

## Output
output/cleaned_marketing_campaign.csv

Note: Outliers were identified, not automatically removed.
"""

with open("output/cleaning_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)

print("\n========== TASK COMPLETED ==========")
print("Cleaned dataset: output/cleaned_marketing_campaign.csv")
print("Summary report: output/cleaning_summary.md")

display(df.head())


========== TASK COMPLETED ==========
Cleaned dataset: output/cleaned_marketing_campaign.csv
Summary report: output/cleaning_summary.md


,id,year_birth,education,marital_status,income,kidhome,teenhome,dt_customer,recency,mntwines,...,numwebvisitsmonth,acceptedcmp3,acceptedcmp4,acceptedcmp5,acceptedcmp1,acceptedcmp2,complain,z_costcontact,z_revenue,response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,Phd,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0
